# Milestone 2: Data Quality dan Profiling

Lanjutan dari `01_data_profiling.ipynb`. Notebook ini menerapkan profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing values, outlier, duplikat, serta validasi data pada **Indonesian News Dataset**.

Setiap keputusan cleaning dicatat dalam *cleaning log* (jumlah baris sebelum dan sesudah) agar prosesnya terdokumentasi.

## 0. Konfigurasi dan Persiapan Data

Dataset mentah (CSV) dikonversi sekali ke **Parquet** tanpa kolom `embedding` (vektor besar yang tidak dipakai analisis). Parquet jauh lebih cepat dan hemat ruang untuk Polars dan DuckDB.

In [1]:
%pip install duckdb pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from datetime import date
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":      # notebook dijalankan dari folder notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "raw" / "data.csv"          # sesuaikan jika file CSV ada di tempat lain
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "dataset.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_bersih.parquet"

print("Project root :", PROJECT_ROOT)
print("CSV ada      :", CSV_PATH.exists())
print("Parquet ada  :", DATA_PATH.exists())

Project root : D:\1. UMM\SEMESTER 7\ANALISIS BIG DATA\TUGAS 1\tugas1-abd-2026
CSV ada      : True
Parquet ada  : False


In [6]:
if not DATA_PATH.exists():
    if not CSV_PATH.exists():
        raise FileNotFoundError(
            f"Dataset tidak ditemukan: {CSV_PATH}. Letakkan data.csv di data/raw/ atau ubah CSV_PATH."
        )
    (
        pl.scan_csv(CSV_PATH, infer_schema_length=0)
        .drop("embedding")
        .with_columns(
            pl.col("id").cast(pl.Int64, strict=False),
            pl.col("date").str.slice(0, 10).str.to_date("%Y-%m-%d", strict=False),
        )
        .sink_parquet(DATA_PATH)       # jika error, ganti dengan .collect().write_parquet(DATA_PATH)
    )
    print("Konversi CSV -> Parquet selesai.")

df = pl.read_parquet(DATA_PATH)
print(f"Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom")
df.select(["id", "source", "title", "date"]).head()

Konversi CSV -> Parquet selesai.
Ukuran awal: 32,735 baris x 10 kolom


id,source,title,date
i64,str,str,date
83,"""tempo""","""Depo Plumpang Terbakar, Anggot…",2023-03-04
84,"""tempo""","""Jokowi Perintahkan Wapres Ma'r…",2023-03-04
85,"""tempo""","""HNW Mendukung Jamaah Umroh Fir…",2023-03-04
86,"""tempo""","""Tim Dokkes Polri Telah Terima …",2023-03-04
87,"""tempo""","""Bamsoet Ajak Komunitas Otomoti…",2023-03-04


## 1. Profiling dengan DuckDB `SUMMARIZE` (Sebelum Cleaning)

`SUMMARIZE` dijalankan langsung pada file Parquet. Kolom teks panjang (`content`, `summary`) diwakili panjang karakternya agar profiling tetap cepat.

In [8]:
import duckdb
print(duckdb.__version__)

2.0.0


In [10]:
connection = duckdb.connect()

def query_pl(sql: str) -> pl.DataFrame:
    """Jalankan query DuckDB dan kembalikan hasilnya sebagai DataFrame Polars."""
    rel = connection.sql(sql)
    try:
        return rel.pl()
    except AttributeError:
        for nama in ("to_arrow_table", "fetch_arrow_table", "arrow"):
            if hasattr(rel, nama):
                return pl.from_arrow(getattr(rel, nama)())
        raise

def summarize_parquet(path):
    return query_pl(f"""
        SUMMARIZE
        SELECT id, source, title, url, "date",
               length(content) AS panjang_isi,
               length(summary) AS panjang_ringkasan
        FROM read_parquet('{Path(path).as_posix()}')
    """)

profil_sebelum = summarize_parquet(DATA_PATH)
profil_sebelum

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32590,"""34081.74877042921""","""18044.068638414126""","""17745""","""35146""","""49818""",32735,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",7,null,null,null,null,null,32735,0.00
"""title""","""VARCHAR""",""" Daftar Panglima Kostrad Dari …","""​​Tips Atasi Masalah Jerawat d…",32823,null,null,null,null,null,32735,0.00
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",34771,null,null,null,null,null,32735,0.00
"""date""","""DATE""","""2018-08-10""","""2023-04-11""",41,"""2023-03-26 02:18:07.643195""",null,"""2023-03-17""","""2023-03-26""","""2023-04-04""",32735,0.00
"""panjang_isi""","""BIGINT""","""0""","""13833""",5077,"""2266.7978310676644""","""1212.2226288522904""","""1514""","""2071""","""2775""",32735,0.00
"""panjang_ringkasan""","""BIGINT""","""59""","""5724""",717,"""446.03810542398776""","""116.24265547989353""","""369""","""451""","""525""",32735,0.03


## 2. Deteksi Missing Values

Selain `null`, dihitung juga string kosong atau hanya spasi, karena data mentah dibaca dari CSV.

In [11]:
jumlah_baris = len(df)
str_cols = [c for c, t in df.schema.items() if t == pl.String]
kosong = {c: df.select((pl.col(c).str.strip_chars() == "").sum()).item() for c in str_cols}

null_summary = (
    pl.DataFrame({
        "kolom": df.columns,
        "jumlah_null": [df[c].null_count() for c in df.columns],
        "string_kosong": [kosong.get(c, 0) for c in df.columns],
    })
    .with_columns(
        ((pl.col("jumlah_null") + pl.col("string_kosong")) / jumlah_baris * 100)
        .round(2)
        .alias("persen_kosong")
    )
    .sort("persen_kosong", descending=True)
)
null_summary

kolom,jumlah_null,string_kosong,persen_kosong
str,i64,i64,f64
"""content""",0,441,1.35
"""image""",0,160,0.49
"""summary""",10,0,0.03
"""id""",0,0,0.0
"""source""",0,0,0.0
"""title""",0,0,0.0
"""url""",0,0,0.0
"""date""",0,0,0.0
"""created_at""",0,0,0.0


## 3. Penanganan Missing Values

Strategi (dapat disesuaikan setelah melihat hasil bagian 2):

- String kosong/spasi dinormalisasi menjadi `null`.
- `url`, `title`, `content`, dan `date` adalah kolom inti. Baris tanpa salah satunya dibuang, karena tanpa itu artikel tidak bisa diidentifikasi atau dianalisis (analisis temporal butuh `date`).
- `source` yang kosong diisi `tidak diketahui`.
- `summary` dan `image` yang kosong dibiarkan, karena tidak dipakai sebagai kunci analisis.

In [12]:
log = []

def catat(langkah, sebelum, sesudah, alasan):
    log.append({"langkah": langkah, "baris_sebelum": sebelum, "baris_sesudah": sesudah,
                "baris_dibuang": sebelum - sesudah, "alasan": alasan})

# 1) Normalisasi string kosong -> null, rapikan spasi judul
df_missing_handled = df.with_columns([
    pl.when(pl.col(c).str.strip_chars() == "").then(None).otherwise(pl.col(c)).alias(c)
    for c in str_cols
]).with_columns(
    pl.col("title").str.strip_chars(),
    pl.col("source").fill_null("tidak diketahui"),
)

# 2) Buang baris yang kolom intinya kosong
sebelum = len(df_missing_handled)
df_missing_handled = df_missing_handled.filter(
    pl.col("url").is_not_null()
    & pl.col("title").is_not_null()
    & pl.col("content").is_not_null()
    & pl.col("date").is_not_null()
)
catat("Buang baris dengan url/title/content/date kosong", sebelum, len(df_missing_handled),
      "Kolom inti tidak boleh kosong")

print(f"Null sebelum: {df.null_count().sum_horizontal().sum():,}")
print(f"Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}")
df_missing_handled.null_count()

Null sebelum: 10
Null sesudah: 167


id,source,title,image,url,content,date,created_at,updated_at,summary
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,157,0,0,0,0,0,10


## 4. Deteksi dan Penanganan Outlier

Kolom numerik yang bermakna pada data berita adalah **panjang isi artikel**. Metode IQR dipakai untuk mendeteksi, lalu dikombinasikan dengan aturan domain:

- Artikel yang sangat panjang bisa jadi liputan mendalam yang sah, sehingga hanya **ditandai** (`outlier_panjang_isi`), tidak dibuang.
- Artikel yang terlalu pendek bukan artikel utuh (potongan atau halaman error), sehingga dibuang.
- Tanggal di luar rentang wajar (sebelum 2000 atau di masa depan) dibuang.

In [13]:
df_outlier_handled = df_missing_handled.with_columns(
    pl.col("content").str.len_chars().alias("panjang_isi"),
    pl.col("title").str.len_chars().alias("panjang_judul"),
)

persentil = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
df_outlier_handled.select(
    [pl.col("panjang_isi").quantile(q).alias(f"p{int(q * 100)}") for q in persentil]
)

p1,p5,p25,p50,p75,p95,p99
f64,f64,f64,f64,f64,f64,f64
343.0,830.0,1536.0,2087.0,2789.0,4412.0,6568.0


In [14]:
KOLOM_NUMERIK = "panjang_isi"

q1 = df_outlier_handled[KOLOM_NUMERIK].quantile(0.25)
q3 = df_outlier_handled[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = max(q1 - 1.5 * iqr, 0)
batas_atas = q3 + 1.5 * iqr

print(f"Q1={q1}, Q3={q3}, IQR={iqr}")
print(f"Batas bawah: {batas_bawah}, Batas atas: {batas_atas}")

df_outlier_handled = df_outlier_handled.with_columns(
    ((pl.col(KOLOM_NUMERIK) < batas_bawah) | (pl.col(KOLOM_NUMERIK) > batas_atas))
    .alias("outlier_panjang_isi")
)
print("Outlier terdeteksi (ditandai):", df_outlier_handled["outlier_panjang_isi"].sum())

# Lihat contoh artikel pendek sebelum menentukan batas minimum
df_outlier_handled.filter(pl.col("panjang_isi") < 300).select(["source", "title", "panjang_isi"]).head(10)

Q1=1536.0, Q3=2789.0, IQR=1253.0
Batas bawah: 0, Batas atas: 4668.5
Outlier terdeteksi (ditandai): 1311


source,title,panjang_isi
str,str,u32
"""cnbcindonesia""","""Intelijen AS Ramal Rusia Sulit…",270
"""cnbcindonesia""","""Video: Intel AS Ungkap Hubunga…",235
"""cnbcindonesia""","""Kejagung Usut Dugaan Korupsi A…",284
"""cnbcindonesia""","""Korut Luncurkan Rudal Jelang L…",226
"""cnbcindonesia""","""Video: PLTN Zaporizhzhia Lumpu…",245
"""cnbcindonesia""","""Video: Terungkap! Iran Pasok S…",212
"""cnbcindonesia""","""Cetak Sejarah, Xi Jinping Jadi…",215
"""cnbcindonesia""","""Video: Keluarkan Aturan Bulan …",289
"""cnbcindonesia""","""Video: Pilot Susi Air Bebas""",269


In [15]:
MIN_PANJANG_ISI = 200          # sesuaikan setelah melihat contoh di atas
TANGGAL_MIN = date(2000, 1, 1)
TANGGAL_MAX = date.today()

sebelum = len(df_outlier_handled)
df_outlier_handled = df_outlier_handled.filter(pl.col("panjang_isi") >= MIN_PANJANG_ISI)
catat(f"Buang artikel dengan isi < {MIN_PANJANG_ISI} karakter", sebelum, len(df_outlier_handled),
      "Bukan artikel utuh")

sebelum = len(df_outlier_handled)
df_outlier_handled = df_outlier_handled.filter(pl.col("date").is_between(TANGGAL_MIN, TANGGAL_MAX))
catat(f"Buang tanggal di luar {TANGGAL_MIN} s.d. {TANGGAL_MAX}", sebelum, len(df_outlier_handled),
      "Tanggal tidak wajar")
print(f"Baris tersisa: {len(df_outlier_handled):,}")

Baris tersisa: 32,287


## 5. Deduplikasi

Kunci bisnis utama adalah `url` (satu artikel = satu URL). Deduplikasi tidak dilakukan pada seluruh baris. Setelah itu dicek duplikat dengan kombinasi `source + title + date`, yaitu artikel sama yang tersimpan dengan URL berbeda.

In [16]:
KOLOM_KUNCI = "url"

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique()
print(f"Total baris: {total_baris:,}")
print(f"Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}")
print(f"Duplikat {KOLOM_KUNCI}: {total_baris - baris_unik:,}")
print(f"ID unik: {df_outlier_handled['id'].n_unique():,} dari {total_baris:,}")

df_dedup = df_outlier_handled.unique(subset=[KOLOM_KUNCI], keep="first", maintain_order=True)
catat(f"Hapus duplikat berdasarkan {KOLOM_KUNCI}", total_baris, len(df_dedup), "Satu artikel satu URL")

sebelum = len(df_dedup)
df_dedup = df_dedup.unique(subset=["source", "title", "date"], keep="first", maintain_order=True)
catat("Hapus duplikat source+title+date", sebelum, len(df_dedup),
      "Artikel sama dengan URL berbeda")
print(f"Baris setelah deduplikasi: {len(df_dedup):,}")

Total baris: 32,287
Baris unik berdasarkan url: 32,287
Duplikat url: 0
ID unik: 32,287 dari 32,287
Baris setelah deduplikasi: 32,273


## 6. Validasi Data Dasar

Aturan kualitas diuji dengan SQL DuckDB pada data bersih. Hasil yang diharapkan: **semua pelanggaran = 0**.

In [19]:
try:
    connection.close()
except Exception:
    pass

connection = duckdb.connect()

In [20]:
connection.register("dataset_bersih", df_dedup)

validasi = query_pl(f"""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(*) FILTER (WHERE title IS NULL)   AS pelanggaran_judul_null,
        COUNT(*) FILTER (WHERE content IS NULL) AS pelanggaran_isi_null,
        COUNT(*) FILTER (WHERE "date" IS NULL)  AS pelanggaran_tanggal_null,
        COUNT(*) FILTER (WHERE "date" < DATE '{TANGGAL_MIN}' OR "date" > DATE '{TANGGAL_MAX}')
                                                AS pelanggaran_rentang_tanggal,
        COUNT(*) FILTER (WHERE panjang_isi < {MIN_PANJANG_ISI}) AS pelanggaran_isi_pendek,
        COUNT(*) FILTER (WHERE url IS NULL OR url NOT LIKE 'http%') AS pelanggaran_url,
        COUNT(*) - COUNT(DISTINCT url)          AS pelanggaran_duplikat_url
    FROM dataset_bersih
""")
validasi

total_baris,pelanggaran_judul_null,pelanggaran_isi_null,pelanggaran_tanggal_null,pelanggaran_rentang_tanggal,pelanggaran_isi_pendek,pelanggaran_url,pelanggaran_duplikat_url
i64,i64,i64,i64,i64,i64,i64,i64
32273,0,0,0,0,0,0,0


## 7. Simpan Data Bersih

Hasil disimpan ke `data/processed/` agar dipakai pada Milestone 3 tanpa mengulang cleaning.

In [21]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_dedup.write_parquet(OUTPUT_PATH)

print(f"Baris sebelum cleaning: {len(df):,}")
print(f"Baris sesudah cleaning: {len(df_dedup):,}")
print(f"Data bersih tersimpan di: {OUTPUT_PATH}")

Baris sebelum cleaning: 32,735
Baris sesudah cleaning: 32,273
Data bersih tersimpan di: D:\1. UMM\SEMESTER 7\ANALISIS BIG DATA\TUGAS 1\tugas1-abd-2026\data\processed\dataset_bersih.parquet


In [22]:
# Cleaning log: rangkuman setiap langkah
pl.DataFrame(log)

langkah,baris_sebelum,baris_sesudah,baris_dibuang,alasan
str,i64,i64,i64,str
"""Buang baris dengan url/title/c…",32735,32294,441,"""Kolom inti tidak boleh kosong"""
"""Buang artikel dengan isi < 200…",32294,32287,7,"""Bukan artikel utuh"""
"""Buang tanggal di luar 2000-01-…",32287,32287,0,"""Tanggal tidak wajar"""
"""Hapus duplikat berdasarkan url""",32287,32287,0,"""Satu artikel satu URL"""
"""Hapus duplikat source+title+da…",32287,32273,14,"""Artikel sama dengan URL berbed…"


In [25]:
try:
    connection.close()
except Exception:
    pass

connection = duckdb.connect()

In [26]:
profil_sesudah = query_pl(f"""
    SUMMARIZE
    SELECT id, source, title, url, "date", panjang_isi,
           length(summary) AS panjang_ringkasan
    FROM read_parquet('{OUTPUT_PATH.as_posix()}')
""")
profil_sesudah

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,str,str,str,str,str,i64,"decimal[9,2]"
"""id""","""BIGINT""","""33""","""63567""",32107,"""34096.70789824311""","""18015.309578226836""","""17761""","""35058""","""49799""",32273,0.00
"""source""","""VARCHAR""","""cnbcindonesia""","""tempo""",7,null,null,null,null,null,32273,0.00
"""title""","""VARCHAR""","""""Aku Iki Dudu Debt Collector"" …","""​​Tips Atasi Masalah Jerawat d…",32249,null,null,null,null,null,32273,0.00
"""url""","""VARCHAR""","""https://bisnis.tempo.co/read/1…","""https://www.suara.com/news/202…",34499,null,null,null,null,null,32273,0.00
"""date""","""DATE""","""2018-08-10""","""2023-04-11""",41,"""2023-03-26 02:31:34.314133""",null,"""2023-03-17""","""2023-03-26""","""2023-04-04""",32273,0.00
"""panjang_isi""","""UINTEGER""","""206""","""13833""",5077,"""2298.343444984972""","""1190.6295232888654""","""1537""","""2087""","""2788""",32273,0.00
"""panjang_ringkasan""","""BIGINT""","""89""","""985""",704,"""446.79908873942287""","""110.76912341012687""","""370""","""452""","""526""",32273,0.03


## Catatan Milestone 2

*Isi dengan kata-kata Anda sendiri setelah menjalankan seluruh sel:*

- **Dimensi kualitas yang paling bermasalah** (kelengkapan, keunikan, validitas, atau konsistensi): Kelengkapan ( completeness ). Terdapat 441 baris pada kolom content yang berisi string kosong (bertipe string_kosong), serta 10 missing values (null) pada kolom summary.
- **Strategi missing values** dan alasannya: Baris yang memiliki missing value atau string kosong pada kolom-kolom inti (url/title/content) dibuang. Alasannya, artikel berita tanpa isi tidak dapat memberikan informasi yang valid untuk dianalisis dan tidak bisa direkonstruksi secara otomatis.
- **Strategi outlier** (mengapa ditandai vs dibuang, alasan batas `MIN_PANJANG_ISI`): Outlier berupa panjang isi artikel yang sangat pendek (di bawah batas MIN_PANJANG_ISI yaitu 200 karakter) dibuang sebanyak 7 baris. Hal ini dilakukan karena teks yang terlalu pendek biasanya bukan artikel berita utuh (misalnya hanya berupa caption foto, judul berulang, atau error page). Sementara itu, data pada batas persentil atas tidak dibuang karena artikel berita memang wajar memiliki teks yang sangat panjang.
- **Jumlah baris yang berubah** (dari cleaning log) dan persentasenya: Total baris berkurang sebanyak 462 baris (dari 32.735 baris menjadi 32.273 baris), atau berkurang sekitar 1,41%.
- **Aturan validasi** yang diterapkan dan hasilnya: Aturan validasi mencakup pengecekan nilai null pada judul/isi/tanggal, pengecekan rentang tanggal yang wajar (tahun 2000-01-01 hingga sekarang), pengecekan isi teks pendek, format URL, serta duplikasi URL. Hasil akhir validasi menunjukkan 0 pelanggaran pada seluruh indikator (sesemua bernilai 0 untuk 32.273 baris).
- **Perbandingan SUMMARIZE sebelum dan sesudah** (perubahan min/max, null, distinct):
    - Min/Max: Nilai minimum panjang_isi naik dari 0 menjadi 206 (karena penghapusan isi di bawah 200 karakter). Nilai maksimum panjang_ringkasan turun dari 5724 menjadi 985.   
    - Null: Persentase null untuk kolom inti tetap 0.00%, sedangkan panjang_ringkasan konsisten berada di angka 0.03%.   
    - Distinct (approx_unique): Nilai unik id berkurang dari 32.590 menjadi 32.107, title berkurang dari 32.823 menjadi 32.249, dan url berkurang dari 34.771 menjadi 34.499 akibat pembersihan baris invalid dan duplikat.   